In [1]:
# Cell 1
!pip install easyocr rapidfuzz textblob gspread ultralytics
!pip install transformers accelerate open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00


In [2]:
# Cell 2
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os

def auto_extract_ads(target_url):
    # 1. Fetch the webpage
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(target_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    # 2. Find images (filtering for common ad containers or large images)
    image_list = []
    for img in soup.find_all('img'):
        src = img.get('src')
        if src:
            full_url = urljoin(target_url, src)
            # Optional: Add logic to filter by size or 'ad' keywords
            image_list.append(full_url)

    return image_list

#Usage:
# website_url = "https://example-shopping-site.com"
# ads = auto_extract_ads(website_url)
# # Then loop through 'ads' and run your calculate_final_score() function!

In [3]:
# Cell 3
import os
import re
import cv2
import pandas as pd
import easyocr
from rapidfuzz import fuzz
from textblob import Word
import gspread
from google.colab import auth
from google.auth import default
from sklearn.metrics import classification_report
from ultralytics import YOLO

from google.colab import drive
drive.mount('/content/drive')

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


In [4]:
# Cell 4
FOLDER_PATH = "/content/drive/MyDrive/Technology-dataset/ads_project/raw_images/ads-ds/"
GOOGLE_SHEET_NAME = "tech-dataset"
THRESHOLD = 0.60

print("Loading OCR and YOLO Models...")
reader = easyocr.Reader(['en'], gpu=True)
yolo_model = YOLO('yolov8n.pt')
print("Models loaded successfully!")


Loading OCR and YOLO Models...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Models loaded successfully!


In [5]:
# Cell 5
STOPWORDS = {
    "the", "is", "and", "to", "in", "of", "on", "with",
    "this", "that", "from", "your", "now", "buy", "offer",
    "sale", "free", "limited", "shop", "only", "today",
    "a", "an", "as", "at", "be", "by", "do", "for", "if",
    "it", "me", "my", "no", "or", "so", "up", "we", "all",
    "any", "but", "can", "did", "had", "has", "her", "him",
    "his", "how", "i", "just", "may", "out", "say", "see",
    "she", "so", "than", "then", "them", "they", "this", "us",
    "was", "what", "when", "where", "who", "why", "will", "you"
}

YOLO_CLASS_MAPPING = {
    "cell phone": "smart phone",
    "tv": "display",
    "monitor": "display",
    "clock": "watch",
    "earbuds": "headphones"
}

def clean_text(text):
    text = text.lower() #converts it to lowercase
    text = re.sub(r'[^a-zA-Z ]', ' ', text) # remove any characters that are not letters or spaces
    return text

def correct_word(word):
    try:
        return str(Word(word).correct()) #attempts to correct the spelling of a given word [textblob.Word().correct() method]
    except:
        return word #original word if an error occurs


In [6]:
# Cell 6 : OCR & YOLO

# MAIN FUNCTION for extracting text-based keywords from an image using OCR
def extract_ocr_keywords(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return set()

    # converts it to grayscale
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, img_bin = cv2.threshold(img_gray, 150, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU) # applies a binary threshold to enhance text visibility for OCR

    results = reader.readtext(img_bin)
    text = " ".join([res[1] for res in results])
    clean_words = clean_text(text).split()

    keywords = set()
    for word in clean_words:
        if len(word) > 2 and word not in STOPWORDS: # keeping only those longer than 2 characters and not present in the STOPWORDS
            keywords.add(correct_word(word)) # corrects their spelling before returning them as a set of unique keywords
    return keywords


# MAIN FUNCTION for detecting objects within an image using the YOLO (You Only Look Once) model

def extract_yolo_objects(image_path):
    results = yolo_model(image_path, verbose=False)
    detected_objects = set()

    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            class_name = yolo_model.names[cls_id].lower()

            mapped_name = YOLO_CLASS_MAPPING.get(class_name, class_name)
            detected_objects.add(mapped_name)

    return detected_objects


In [7]:
# Cell 7

# Function calculates a similarity score
# between a set of found_keywords (extracted from an image) and a set of expected_keywords (provided by an advertiser)

# expected_word are averaged to produce a final similarity score, normalized between 0 and 1

def fuzzy_match_score(found_keywords, expected_keywords):
    score = 0
    total_matches = 0

    for expected_word in expected_keywords:
        expected_word = expected_word.strip().lower()
        best_match = 0

        for found_word in found_keywords:
            similarity = fuzz.partial_ratio(expected_word, found_word) / 100 # best fuzzy matching similarity
            best_match = max(best_match, similarity)

        score += best_match
        total_matches += 1

    if total_matches == 0:
        return 0

    return score / total_matches

def classify_ad(score):
    confidence_score = round(score * 100, 2)
    if score >= 0.60:
        return f"Ad is {confidence_score}% likely to be AUTHENTIC", "success"
    elif score >= 0.45:
        return f"Ad is {confidence_score}% likely to be SUSPICIOUS", "warning"
    else:
        return f"Ad is {round(100 - confidence_score, 2)}% likely to be MISMATCH", "error"

In [8]:
# Cell 8

import torch  # for GPU acceleration
from PIL import Image # for image handling
from transformers import BlipProcessor, BlipForConditionalGeneration # for the BLIP model
import open_clip # for the CLIP model

# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print("Loading BLIP model (image captioning)...")

# loads the BLIP (Bootstrapping Language-Image Pre-training) model
# to handle image preprocessing and text tokenization for the BLIP model
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")

# for tasks like image captioning, and it's moved to the selected device (GPU or CPU)
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to(device)
print("BLIP model loaded successfully!")

print("Loading CLIP model (image-text similarity)...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model.to(device)
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')
print("CLIP model loaded successfully!")

Using device: cuda
Loading BLIP model (image captioning)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-large
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BLIP model loaded successfully!
Loading CLIP model (image-text similarity)...


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP model loaded successfully!


In [9]:
# Cell 9

def generate_image_caption(image_path):
    try:
        raw_image = Image.open(image_path).convert('RGB') # converts it to 'RGB' format
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return None

    # Preprocess the image for the BLIP model
    inputs = blip_processor(raw_image, return_tensors="pt").to(device)

    # Generate the caption
    out = blip_model.generate(**inputs, max_length=50, num_beams=4) #uses the BLIP model to predict a sequence of tokens that form the image caption. num_beams=4 uses beam search with 4 beams to find a more likely caption

    # Decode the generated caption
    caption = blip_processor.decode(out[0], skip_special_tokens=True) # the generated token IDs (out[0]) are decoded back into a human-readable string
    return caption


In [10]:
# Cell 10
def calculate_clip_similarity(image_path, text_queries):
    try:
        raw_image = Image.open(image_path).convert('RGB')
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return 0.0 # Return 0 similarity if image not found

    # Preprocess and encode image
    image_input = clip_preprocess(raw_image).unsqueeze(0).to(device)
    with torch.no_grad():
        image_features = clip_model.encode_image(image_input)

    # Tokenize and encode text queries
    text_tokens = clip_tokenizer(text_queries).to(device)
    with torch.no_grad():
        text_features = clip_model.encode_text(text_tokens)

    # Normalize features
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Calculate cosine similarity
    # The result is already cosine similarity because features are normalized
    similarity_scores = (image_features @ text_features.T).squeeze(0)

    # Return the maximum similarity if comparing against multiple text queries
    return similarity_scores.max().item()

In [11]:
def calculate_final_score(image_path, advertiser_keywords, ocr_keywords, yolo_keywords, blip_caption, clip_sim):
    # 2. Calculate ocr_score
    ocr_score = fuzzy_match_score(ocr_keywords, advertiser_keywords)

    # 3. Calculate yolo_score
    yolo_score = fuzzy_match_score(yolo_keywords, advertiser_keywords)

    # 4. Use the provided blip_caption
    caption = blip_caption

    # 5. Use the provided clip_sim
    clip_image_advertiser_similarity = clip_sim

    # 6. Calculate caption_score
    caption_score = 0.0
    if caption and caption.strip():
        caption_score = fuzzy_match_score(set(caption.split()), advertiser_keywords)

    # 7. Calculate the final_score as a weighted sum
    final_score = (
    (ocr_score * 0.45) +
    (yolo_score * 0.05) +
    (clip_image_advertiser_similarity * 0.30) +
    (caption_score * 0.20)
    )

    # 8. Return the final_score
    return final_score

In [12]:
# Cell 12: Integrated Multi-Modal Analysis
sheet = gc.open(GOOGLE_SHEET_NAME)
worksheet = sheet.get_worksheet(0)
df = pd.DataFrame(worksheet.get_all_records())

y_true = []
y_pred = []

print("🚀 Starting Integrated Multi-Modal Authenticity Analysis...\n")

for index, row in df.iterrows():
    image_name = row["Image_Name"]
    image_path = os.path.join(FOLDER_PATH, image_name)

    # Get Expected Keywords
    advertiser_keywords = set([k.strip().lower() for k in str(row["Advertiser_Keywords"]).split(",")])
    actual_label = row["Expected_Label"]

    if not os.path.exists(image_path):
        print(f"❌ Missing Image File: {image_name}")
        continue

    # --- INTEGRATED SCORING ---
    # 1. Extract base features
    ocr_keywords = extract_ocr_keywords(image_path)
    yolo_keywords = extract_yolo_objects(image_path)
    blip_caption = generate_image_caption(image_path)
    clip_sim = calculate_clip_similarity(image_path, list(advertiser_keywords))

    # 2. Get the weighted final score (Includes CLIP and BLIP)
    # Pass blip_caption and clip_sim as they are already calculated
    score = calculate_final_score(image_path, advertiser_keywords, ocr_keywords, yolo_keywords, blip_caption, clip_sim)

    # 3. Classify based on the new weighted score
    # Unpack the tuple: (message, status)
    raw_prediction, status_type = classify_ad(score)
    # --------------------------

    # Map to "GOOD" or "MISLEADING"
    if "AUTHENTIC" in raw_prediction:
        display_prediction = 'GOOD'
    elif "SUSPICIOUS" in raw_prediction or "MISMATCH" in raw_prediction:
        display_prediction = 'MISLEADING'
    else:
        display_prediction = raw_prediction

    y_true.append(actual_label)
    y_pred.append(display_prediction)

    # Detailed Output
    print("=================================")
    print(f"Image: {image_name}")
    print(f"🔍 OCR: {list(ocr_keywords)[:5]}")
    print(f"📸 YOLO: {list(yolo_keywords)}")
    print(f"📝 BLIP:  \"{blip_caption}\"")
    print(f"🔗 CLIP:  {round(clip_sim, 2)}")
    print(f"📊 Final Multi-Modal Score: {round(score, 2)}")
    print(f"🤖 Prediction: {display_prediction} | 🎯 Actual: {actual_label}")

# Final Report
print("\n" + "="*35)
print("🏆 FINAL SYSTEM RESULTS (MULTI-MODAL) 🏆")
print("="*35)
accuracy = sum([1 for t, p in zip(y_true, y_pred) if t == p]) / len(y_true)
print(f"Overall Accuracy: {round(accuracy * 100, 2)}%\n")
print("Classification Report:")
print(classification_report(y_true, y_pred))



🚀 Starting Integrated Multi-Modal Authenticity Analysis...

Image: camera_1.jpg
🔍 OCR: ['reallygreatsite', 'off', 'camera']
📸 YOLO: []
📝 BLIP:  "a black and gold poster with a camera on a pedestal"
🔗 CLIP:  0.21
📊 Final Multi-Modal Score: 0.46
🤖 Prediction: MISLEADING | 🎯 Actual: MISLEADING
Image: camera_2.jpg
🔍 OCR: ['long', 'built', 'low', 'limited', 'coapryis']
📸 YOLO: []
📝 BLIP:  "an advertisement for a camera sale with a picture of a camera"
🔗 CLIP:  0.26
📊 Final Multi-Modal Score: 0.49
🤖 Prediction: MISLEADING | 🎯 Actual: MISLEADING
Image: camera_3.jpg
🔍 OCR: ['nosttud', 'god', 'consectatut', 'common', 'term']
📸 YOLO: ['smart phone']
📝 BLIP:  "the black friday sale is on gadgets up to 80 % off"
🔗 CLIP:  0.25
📊 Final Multi-Modal Score: 0.59
🤖 Prediction: MISLEADING | 🎯 Actual: MISLEADING
Image: camera_4.jpg
🔍 OCR: ['in', 'digital', 'log', 'fully', 'here']
📸 YOLO: []
📝 BLIP:  "a purple and yellow poster with a photo of a digital camera"
🔗 CLIP:  0.28
📊 Final Multi-Modal Score: 0.65

In [13]:
import torch
import gradio as gr
import numpy as np
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import open_clip
from rapidfuzz import fuzz

# Add for web scraping and file handling
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import os
import shutil # For removing temporary directories

# --- 1. CONFIGURATION & OPTIMIZATION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SIZE = "ViT-B-32"
PRETRAINED = "openai"

# Predefined prompts for CLIP
AUTHENTICITY_PROMPTS = [
    "genuine branded high quality product advertisement",
    "fake cheap scam product advertisement",
    "low quality misleading sketchy ad"
]

# --- 2. CACHED MODEL LOADING ---
print(f"Loading models on {DEVICE}...")

# Load CLIP
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(MODEL_SIZE, pretrained=PRETRAINED)
clip_model.to(DEVICE).eval()
clip_tokenizer = open_clip.get_tokenizer(MODEL_SIZE)

# Load BLIP-2
blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()

print("Models loaded successfully.")

# --- 3. CORE ENGINE ---

def normalize_keywords(text):
    synonyms = {"phone": "smartphone", "mobile": "smartphone", "pc": "laptop", "computer": "laptop"}
    words = text.lower().replace(",", " ").split()
    return [synonyms.get(w, w) for w in words]

def get_clip_similarity(image_input, text_list):
    if not text_list: return 0.0
    with torch.no_grad():
        tokens = clip_tokenizer(text_list).to(DEVICE)
        text_features = clip_model.encode_text(tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        image_features = clip_model.encode_image(image_input)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # Softmax-like scaling is often better for raw similarity scores
        similarities = (image_features @ text_features.T).squeeze(0)
        return similarities.max().item()

def analyze_ad(image, user_keywords_input):
    # Optimization: Resize
    img_resized = image.resize((224, 224))
    img_tensor = clip_preprocess(img_resized).unsqueeze(0).to(DEVICE)

    # 1. BLIP-2 Caption
    inputs = blip_processor(images=image, return_tensors="pt").to(DEVICE, torch.float16 if DEVICE == "cuda" else torch.float32)
    generated_ids = blip_model.generate(**inputs, max_new_tokens=20)
    caption = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # 2. Keyword Processing
    user_keywords = normalize_keywords(user_keywords_input)

    # 3. CLIP Similarities (Calibrated)
    # CLIP scores usually range 0.2 - 0.4. We shift and scale to 0-1
    raw_prompt_sim = get_clip_similarity(img_tensor, AUTHENTICITY_PROMPTS)
    prompt_sim = np.clip((raw_prompt_sim - 0.15) / 0.25, 0, 1) # This is the Visual Alignment Score

    raw_keyword_sim = get_clip_similarity(img_tensor, user_keywords) if user_keywords else 0.25

    # Add a small boost to raw_keyword_sim if any user keyword is in the BLIP caption
    if user_keywords and any(kw in caption.lower() for kw in user_keywords):
        raw_keyword_sim += 0.05

    keyword_sim = np.clip((raw_keyword_sim - 0.15) / 0.25, 0, 1)

    # 4. Caption similarity
    cap_match_score = 0
    if user_keywords:
        # Calculate similarity for each user keyword against the caption
        cap_match_score = max([fuzz.partial_ratio(kw, caption.lower()) for kw in user_keywords]) / 100
    else:
        cap_match_score = 0.5 # Default if no keywords provided

    # 5. Final Scoring Logic (Weighted)
    final_score = (
        (0.4 * prompt_sim) +
        (0.3 * keyword_sim) +
        (0.3 * cap_match_score)
    )
    confidence = round(float(final_score) * 100, 2)

    # 6. Initial Classification
    if confidence > 60: classification = "✅ AUTHENTIC (The ad text and objects match with the advertiser's claims)"
    elif confidence > 48: classification = "⚠️SUSPICIOUS (Some keywords match, but others are missing or noise is too high. Require manual review.)"
    else: classification = "❌ MISMATCH (There is almost no correlation between the advertiser's keyword and what is actually in the ad image.)"

    suspicious_reasons = []
    num_user_keywords = len(user_keywords)
    matched_keywords_in_caption = [kw for kw in user_keywords if kw in caption.lower()]
    num_matched_keywords_in_caption = len(matched_keywords_in_caption)

    # --- Refined USER-DEFINED LOGIC FOR 'SUSPICIOUS' FLAG ---

    # Calculate Match Ratio for keywords in caption
    match_ratio_caption = num_matched_keywords_in_caption / num_user_keywords if num_user_keywords > 0 else 0.0

    # Condition 1: If user keywords were provided, check for insufficient matches
    if num_user_keywords > 0:
        if num_matched_keywords_in_caption == 0:
            suspicious_reasons.append("No user keywords found in the generated caption.")
        elif num_matched_keywords_in_caption < 2: # Stricter: at least 2 keywords must match from user input
            # If only 0 or 1 keyword matched, and it was classified authentic, it should be suspicious
            if confidence > 60:
                suspicious_reasons.append(f"Only {num_matched_keywords_in_caption} out of {num_user_keywords} user keywords matched. Insufficient keyword evidence for AUTHENTIC classification.")
        elif match_ratio_caption < 0.5: # Less than 50% of user keywords matched in caption
            if confidence > 60:
                suspicious_reasons.append(f"Only {round(match_ratio_caption * 100, 2)}% of user keywords matched in caption (less than 50%). Insufficient keyword evidence for AUTHENTIC classification.")

    # If new suspicious reasons exist AND current classification is not already a definite MISMATCH,
    # force it to SUSPICIOUS.
    if suspicious_reasons and "❌ MISMATCH" not in classification:
        classification = "⚠️SUSPICIOUS (Requires manual review.)"
        # Optionally, adjust confidence downwards if it was originally 'AUTHENTIC'
        if confidence > 60: # If it was initially authentic but now forced suspicious
            confidence = min(confidence, 55.0) # Cap confidence to a 'suspicious' range for display

    # Update reasoning
    current_reasoning = f"Matched Keywords in Caption: {', '.join(matched_keywords_in_caption) if matched_keywords_in_caption else 'None'}. Visual Alignment (prompt_sim): {round(float(prompt_sim), 2)}. Caption Match Ratio: {round(match_ratio_caption, 2)}."
    if suspicious_reasons:
        reasoning = f"{current_reasoning} | SUSPICION REASONS: {'; '.join(suspicious_reasons)}"
    else:
        reasoning = current_reasoning

    return classification, f"{confidence}%", caption, reasoning


# New function for web scraping and analysis
def analyze_website_ads(website_url, user_keywords_input):
    temp_dir = "temp_scraped_images"
    # Ensure the temporary directory is clean before starting
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir, exist_ok=True)

    results_summary = []

    try:
        response = requests.get(website_url, timeout=15)
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
    except requests.exceptions.RequestException as e:
        if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
        return f"Error fetching URL {website_url}: {e}"

    soup = BeautifulSoup(response.text, 'html.parser')
    base_url = urljoin(website_url, "/")

    image_count = 0
    for i, img_tag in enumerate(soup.find_all('img')):
        img_src = img_tag.get('src')
        if not img_src:
            continue

        absolute_img_url = urljoin(base_url, img_src)

        # Basic filter for potentially relevant images (avoiding tiny icons/placeholders)
        if not absolute_img_url.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.webp')) or \
           'data:image' in absolute_img_url:
            continue

        try:
            # Fetch image data with a timeout
            img_data = requests.get(absolute_img_url, timeout=10).content
            img_name = f"image_{i}_{os.path.basename(urlparse(absolute_img_url).path).split('?')[0]}" # Use part of original filename
            if len(img_name) > 100: img_name = img_name[:100] # Prevent very long filenames
            if not img_name.endswith(('.png', '.jpg', '.jpeg', '.gif', '.webp')):
                img_name += '.jpg' # Default extension

            img_path = os.path.join(temp_dir, img_name)
            with open(img_path, 'wb') as handler:
                handler.write(img_data)

            # Analyze the downloaded image
            image_count += 1
            pil_image = Image.open(img_path).convert('RGB')
            classification, confidence, caption, reasoning = analyze_ad(pil_image, user_keywords_input)

            results_summary.append(f"### Image {image_count}: {absolute_img_url}\n" # Use \n for markdown newlines
                                   f"- **Classification:** {classification}\n"
                                   f"- **Confidence:** {confidence}\n"
                                   f"- **Caption:** {caption}\n"
                                   f"- **Reasoning:** {reasoning}\n")

        except requests.exceptions.RequestException as e:
            results_summary.append(f"### Image (from {absolute_img_url}): Error downloading or accessing - {e}\n")
        # except Image.UnidentifiedImageError:
        #     results_summary.append(f"### Image (from {absolute_img_url}): Could not identify image file. Skipping.\n")
        except Exception as e:
            results_summary.append(f"### Image (from {absolute_img_url}): Error processing - {e}\n")

    if image_count == 0:
        results_summary.append("No suitable images found or successfully processed from the provided URL.")

    # Clean up the temporary directory
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    return "\n".join(results_summary)


# --- 4. GRADIO UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️ Ad Authenticity Detector")

    with gr.Tab("Analyze Image Upload"):
        with gr.Row():
            with gr.Column():
                img_input = gr.Image(type="pil", label="Upload Ad Image")
                kw_input = gr.Textbox(label="Keywords", placeholder="e.g., iPhone, Sony, Watch")
                btn = gr.Button("SCAN Uploaded Image", variant="primary")
            with gr.Column():
                out_class = gr.Label(label="Result")
                out_conf = gr.Textbox(label="Score")
                out_cap = gr.Textbox(label="Caption")
                out_reason = gr.Textbox(label="Reasoning")
        btn.click(fn=analyze_ad, inputs=[img_input, kw_input], outputs=[out_class, out_conf, out_cap, out_reason])

    with gr.Tab("Analyze Website Ads"):
        with gr.Row():
            with gr.Column():
                url_input = gr.Textbox(label="Website URL", placeholder="e.g., https://example.com/ads")
                kw_input_web = gr.Textbox(label="Keywords (for website analysis)", placeholder="e.g., iPhone, Sony, Watch")
                btn_web = gr.Button("SCAN Website Ads", variant="primary")
            with gr.Column():
                out_web_results = gr.Markdown(label="Website Analysis Results", elem_id="website-results")
        btn_web.click(fn=analyze_website_ads, inputs=[url_input, kw_input_web], outputs=out_web_results)

demo.launch(share=True)

Loading models on cuda...


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Models loaded successfully.


/tmp/ipykernel_3684/2313578127.py:224: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76ace7cef484b10219.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
